# Week 2b: AG News Embedding Pipeline (Dataset 2, N=1000)
**Project**: Quantum Kernels on CLIP/BERT Embeddings — ICLR Extension

**Dataset 2 of 3**: AG News — 4-class topic classification (World / Sports / Business / Sci/Tech)

**Why AG News:**
- Different task type from SST-2 (topic vs sentiment)
- Balanced 4-class structure (25% each) — good for macro F1
- Well-studied benchmark for fair comparison with prior QML work
- Same BERT encoder kept frozen — isolates the kernel comparison

**Key difference from Week 2 (SST-2):**
- 4 classes → metric is **macro F1** (not binary F1)
- Column name is `text` (not `sentence`)
- SVC handles multiclass natively via one-vs-one — no code change needed

**Runtime**: ~12 min on Colab T4

In [ ]:
!pip install datasets transformers torch scikit-learn numpy --quiet

import numpy as np
import time, os, pickle, shutil, json
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

---
## Step 0: Storage + checkpoint helpers

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

def _persist_and_download(path):
    if os.path.abspath(SAVE_DIR) != os.getcwd():
        dst = os.path.join(SAVE_DIR, os.path.basename(path))
        shutil.copy(path, dst)
        print(f"  saved -> {dst}")
    try:
        from google.colab import files
        files.download(path)
        print(f"  download -> {os.path.basename(path)}")
    except Exception as e:
        print(f"  (download skipped: {type(e).__name__} — saved above)")

def checkpoint(filename, save_fn):
    save_fn(filename); _persist_and_download(filename)

def save_figure(fig, name):
    for ext in ('png', 'pdf'):
        p = f"{name}.{ext}"
        fig.savefig(p, bbox_inches='tight', dpi=150)
        _persist_and_download(p)

def _find(fname):
    dp = os.path.join(SAVE_DIR, fname)
    return dp if os.path.exists(dp) else fname

print("Helpers ready.")

---
## Step 1: Load AG News

AG News has 120k train / 7.6k test samples. We take 1100 train + 250 test
as a buffer to build a clean 1000 / 200 split.

**Classes:** 0=World, 1=Sports, 2=Business, 3=Sci/Tech

In [ ]:
AG_LABELS = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}

print("Loading AG News...")
dataset = load_dataset("fancyzhx/ag_news")

N_TRAIN_EXTRACT = 1100
N_TEST_EXTRACT  = 250

train_data = dataset['train'].shuffle(seed=42).select(range(N_TRAIN_EXTRACT))
test_data  = dataset['test'].shuffle(seed=42).select(range(N_TEST_EXTRACT))

train_labels = np.array([x['label'] for x in train_data])
test_labels  = np.array([x['label'] for x in test_data])

print(f"Extracted: {len(train_data)} train, {len(test_data)} test")
print(f"\nClass distribution (train):")
for i, name in AG_LABELS.items():
    print(f"  {name:<12}: {np.mean(train_labels==i):.1%}")
print(f"\nExample samples:")
for i in range(4):
    print(f"  [{AG_LABELS[train_data[i]['label']]}] {train_data[i]['text'][:70]}...")

---
## Step 2: Extract BERT embeddings — checkpoint immediately after

Same frozen `bert-base-uncased` as Dataset 1 (SST-2). Keeping the encoder
identical means any difference in results is attributable to the dataset
structure, not the embedding model.

In [ ]:
MODEL_NAME = "bert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
bert_model.eval()
print(f"{MODEL_NAME} loaded ({sum(p.numel() for p in bert_model.parameters()):,} params)")

def extract_bert_embeddings(texts, batch_size=32):
    out = []
    nb  = (len(texts) + batch_size - 1) // batch_size
    for i in range(0, len(texts), batch_size):
        batch  = texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True,
                           max_length=128, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            o = bert_model(**inputs)
        out.append(o.last_hidden_state[:, 0, :].cpu().numpy())
        if (i // batch_size + 1) % 5 == 0:
            print(f"  batch {i//batch_size+1}/{nb}")
    return np.vstack(out)

print("\nExtracting train embeddings...")
t0 = time.time()
train_texts = [x['text'] for x in train_data]
train_emb   = extract_bert_embeddings(train_texts)
print(f"  done {time.time()-t0:.0f}s | {train_emb.shape}")

print("\nExtracting test embeddings...")
test_texts = [x['text'] for x in test_data]
test_emb   = extract_bert_embeddings(test_texts)
print(f"  done | {test_emb.shape}")

print("\nCheckpoint 1 — raw BERT embeddings:")
checkpoint('ag_news_bert_raw.npz',
           lambda p: np.savez(p, train_emb=train_emb, test_emb=test_emb,
                              train_labels=train_labels, test_labels=test_labels))

---
## Step 3: PCA + normalize — checkpoint after

In [ ]:
N_QUBITS = 8
pca_dims = [4, 8, 12, 16]
pca_objects = {n: PCA(n_components=n, random_state=42).fit(train_emb) for n in pca_dims}

print("PCA variance explained:")
for n in pca_dims:
    print(f"  {n:>2} dims -> {np.sum(pca_objects[n].explained_variance_ratio_)*100:.1f}%")

pca_8      = pca_objects[N_QUBITS]
X_train_pca = pca_8.transform(train_emb)
X_test_pca  = pca_8.transform(test_emb)

scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train_scaled = scaler.fit_transform(X_train_pca)
X_test_scaled  = np.clip(scaler.transform(X_test_pca), 0, np.pi)

print(f"\nTrain range: [{X_train_scaled.min():.3f}, {X_train_scaled.max():.3f}]")
print(f"Test  range: [{X_test_scaled.min():.3f}, {X_test_scaled.max():.3f}]")

print("\nCheckpoint 2 — PCA + scaler:")
checkpoint('ag_news_pca_scaler.pkl',
           lambda p: pickle.dump({'pca': pca_8, 'scaler': scaler, 'n_qubits': N_QUBITS,
                                  'dataset': 'ag_news', 'n_classes': 4}, open(p, 'wb')))

---
## Step 4: Classical SVM baselines on real 1000 / 200 split

**Important**: metric is **macro F1** for 4-class classification.
Macro F1 averages F1 per class equally — treats all 4 topics fairly.

In [ ]:
N_TRAIN_SVM = 1000
N_TEST_SVM  = 200
assert len(X_train_scaled) >= N_TRAIN_SVM
assert len(X_test_scaled)  >= N_TEST_SVM

idx_tr = np.random.RandomState(42).choice(len(X_train_scaled), N_TRAIN_SVM, replace=False)
idx_te = np.random.RandomState(42).choice(len(X_test_scaled),  N_TEST_SVM,  replace=False)
X_tr, y_tr = X_train_scaled[idx_tr], train_labels[idx_tr]
X_te, y_te = X_test_scaled[idx_te],  test_labels[idx_te]

print(f"Train {X_tr.shape} | Test {X_te.shape}")
print(f"Class balance (train): {[f'{AG_LABELS[i]}={np.mean(y_tr==i):.0%}' for i in range(4)]}")
print()

classifiers = {
    'Linear':     SVC(kernel='linear', C=1.0, random_state=42),
    'RBF':        SVC(kernel='rbf',    C=1.0, gamma='scale', random_state=42),
    'Polynomial': SVC(kernel='poly',   C=1.0, degree=3, coef0=1, random_state=42),
}
results = {}
print(f"{'Kernel':<14} {'Accuracy':>10} {'Macro-F1':>10} {'Time':>8}")
print("-"*46)
for name, clf in classifiers.items():
    t0 = time.time()
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    results[name] = {
        'accuracy':  float(accuracy_score(y_te, y_pred)),
        'f1_macro':  float(f1_score(y_te, y_pred, average='macro')),
        'f1_per_class': [float(f) for f in f1_score(y_te, y_pred, average=None)]
    }
    print(f"{name:<14} {results[name]['accuracy']:>10.4f} "
          f"{results[name]['f1_macro']:>10.4f} {time.time()-t0:>6.1f}s")

# Per-class breakdown for best kernel
best_k = max(results, key=lambda k: results[k]['f1_macro'])
print(f"\nPer-class F1 ({best_k} — best macro-F1):")
for i, fname in AG_LABELS.items():
    print(f"  {fname:<12}: {results[best_k]['f1_per_class'][i]:.4f}")

print("\nCheckpoint 3 — baseline results:")
checkpoint('ag_news_baseline_results.pkl', lambda p: pickle.dump(results, open(p, 'wb')))

---
## Step 5: PCA dimension sweep

In [ ]:
print(f"  {'Dims':>4}  {'Var%':>6}  {'Acc':>8}  {'MacroF1':>10}")
print("-"*36)
for n_dim in pca_dims:
    X_tr_d = pca_objects[n_dim].transform(train_emb[idx_tr])
    X_te_d = pca_objects[n_dim].transform(test_emb[idx_te])
    sc_d   = MinMaxScaler(feature_range=(0, np.pi))
    X_tr_d = sc_d.fit_transform(X_tr_d)
    X_te_d = np.clip(sc_d.transform(X_te_d), 0, np.pi)
    clf_d  = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42).fit(X_tr_d, y_tr)
    yp = clf_d.predict(X_te_d)
    var = np.sum(pca_objects[n_dim].explained_variance_ratio_)*100
    print(f"  {n_dim:>4}  {var:>5.1f}%  "
          f"{accuracy_score(y_te,yp):>8.4f}  "
          f"{f1_score(y_te,yp,average='macro'):>10.4f}")

---
## Step 6: Comparison figure (vs SST-2 baselines)

In [ ]:
# load SST-2 baselines if available for side-by-side comparison
sst2_results = None
try:
    with open(_find('baseline_results.pkl'), 'rb') as f:
        sst2_results = pickle.load(f)
    print("SST-2 baselines loaded for comparison.")
except:
    print("SST-2 baselines not found — showing AG News only.")

kernels = list(results.keys())
ag_accs = [results[k]['accuracy'] for k in kernels]
ag_f1s  = [results[k]['f1_macro'] for k in kernels]

if sst2_results:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    sst2_accs = [sst2_results[k]['accuracy'] for k in kernels]
    sst2_f1s  = [sst2_results[k]['f1'] for k in kernels]
    x = np.arange(len(kernels)); w = 0.35
    axes[0].bar(x-w/2, sst2_accs, w, label='SST-2 (sentiment)', color='#5B8DB8', edgecolor='white')
    axes[0].bar(x+w/2, ag_accs,   w, label='AG News (topic)',    color='#D95F02', edgecolor='white')
    axes[0].set_xticks(x); axes[0].set_xticklabels(kernels)
    axes[0].set_ylim(0.4, 0.9); axes[0].set_title('Accuracy: SST-2 vs AG News')
    axes[0].set_ylabel('Accuracy'); axes[0].legend()
    axes[1].bar(x-w/2, sst2_f1s, w, label='SST-2 binary F1',  color='#5B8DB8', edgecolor='white')
    axes[1].bar(x+w/2, ag_f1s,   w, label='AG News macro-F1', color='#D95F02', edgecolor='white')
    axes[1].set_xticks(x); axes[1].set_xticklabels(kernels)
    axes[1].set_ylim(0.4, 0.9); axes[1].set_title('F1: SST-2 vs AG News')
    axes[1].set_ylabel('F1'); axes[1].legend()
    plt.suptitle('Classical SVM baselines — 8-dim BERT embeddings (N=1000)', y=1.02)
else:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].bar(kernels, ag_accs, color=['steelblue','coral','mediumseagreen'], edgecolor='white')
    axes[0].set_ylim(0.4, 0.9); axes[0].set_title('AG News Accuracy'); axes[0].set_ylabel('Accuracy')
    for i,v in enumerate(ag_accs): axes[0].text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=11)
    axes[1].bar(kernels, ag_f1s, color=['steelblue','coral','mediumseagreen'], edgecolor='white')
    axes[1].set_ylim(0.4, 0.9); axes[1].set_title('AG News Macro-F1'); axes[1].set_ylabel('Macro-F1')
    for i,v in enumerate(ag_f1s): axes[1].text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=11)
    plt.suptitle('Classical baselines — 8-dim BERT, AG News (N=1000)')
plt.tight_layout(); plt.show()
save_figure(fig, 'ag_news_fig1_baselines')

---
## Step 7: Save final embeddings — with shape verification

In [ ]:
def save_final(p):
    np.savez(p,
             X_train=X_train_scaled, X_test=X_test_scaled,
             y_train=train_labels,   y_test=test_labels,
             X_tr_subset=X_tr, X_te_subset=X_te,
             y_tr_subset=y_tr, y_te_subset=y_te,
             class_names=np.array(list(AG_LABELS.values())))

print("Checkpoint 4 — final embeddings:")
checkpoint('ag_news_embeddings.npz', save_final)

loaded = np.load('ag_news_embeddings.npz')
print(f"\n  X_tr_subset = {loaded['X_tr_subset'].shape}  (must be (1000, 8))")
print(f"  X_te_subset = {loaded['X_te_subset'].shape}  (must be (200, 8))")
assert loaded['X_tr_subset'].shape == (1000, 8), "WRONG TRAIN SIZE"
assert loaded['X_te_subset'].shape == (200, 8),  "WRONG TEST SIZE"
print("\n  VERIFIED: genuine N=1000 / 200 split.")

# Also save a combined metadata file
meta = {
    'dataset': 'ag_news',
    'n_classes': 4,
    'class_names': AG_LABELS,
    'n_train': 1000, 'n_test': 200,
    'n_qubits': N_QUBITS,
    'model': MODEL_NAME,
    'metric': 'macro_f1',
    'baselines': results
}
checkpoint('ag_news_meta.json', lambda p: json.dump(meta, open(p,'w'), indent=2))

print(f"\nAll files in {SAVE_DIR}:")
ag_files = [f for f in sorted(os.listdir(SAVE_DIR)) if 'ag_news' in f]
for f in ag_files:
    sz = os.path.getsize(os.path.join(SAVE_DIR, f))/1024
    print(f"  {f}  ({sz:.0f} KB)")

---
## Week 2b complete

### Saved + downloaded:
- `ag_news_bert_raw.npz` — raw 768-dim BERT embeddings (checkpoint after extraction)
- `ag_news_pca_scaler.pkl` — PCA + scaler objects
- `ag_news_baseline_results.pkl` — 3 classical SVM baselines
- `ag_news_embeddings.npz` — final 1000/200 split, ready for Week 3b
- `ag_news_meta.json` — dataset metadata
- `ag_news_fig1_baselines.png/.pdf` — baseline comparison figure

### Important for Week 3b (quantum kernel on AG News):
- Load `ag_news_embeddings.npz` — same structure as `embeddings_week2.npz`
- Use `f1_score(..., average='macro')` instead of binary F1
- The quantum kernel SVM handles 4-class automatically — no changes needed

### Recovering from a crashed session:
If BERT extraction crashes, reload from checkpoint:
```python
raw = np.load('/content/drive/MyDrive/qml_project/ag_news_bert_raw.npz')
train_emb, test_emb = raw['train_emb'], raw['test_emb']
train_labels, test_labels = raw['train_labels'], raw['test_labels']
```

### What comes after:
- Week 3b: quantum kernel on AG News (same IQP circuit, 4-class multiclass SVM)
- Week 2c: CLIP + CIFAR-10 (vision dataset — different encoder, same quantum pipeline)
- Paper comparison: SST-2 vs AG News vs CIFAR-10 across all methods